In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import datetime as dt

# Load raw transactional dataset
df = pd.read_csv('../data/online_retail.csv', encoding='ISO-8859-1')

# Inspect shape and basic summary
print("Raw Dataset Shape:", df.shape)
df.info()
df.head()

Raw Dataset Shape: (541909, 8)
<class 'pandas.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  str    
 1   StockCode    541909 non-null  str    
 2   Description  540455 non-null  str    
 3   Quantity     541909 non-null  int64  
 4   InvoiceDate  541909 non-null  str    
 5   UnitPrice    541909 non-null  float64
 6   CustomerID   406829 non-null  float64
 7   Country      541909 non-null  str    
dtypes: float64(2), int64(1), str(5)
memory usage: 33.1 MB


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [4]:
# 1. Drop rows missing CustomerID
df_clean = df.dropna(subset=['CustomerID']).copy()

# 2. Convert CustomerID to integer
df_clean['CustomerID'] = df_clean['CustomerID'].astype(int)

# 3. Filter out cancelled transactions (Quantity <= 0) and bad pricing (UnitPrice <= 0)
df_clean = df_clean[(df_clean['Quantity'] > 0) & (df_clean['UnitPrice'] > 0)]

# 4. Convert InvoiceDate to datetime format
df_clean['InvoiceDate'] = pd.to_datetime(df_clean['InvoiceDate'])

# 5. Create TotalAmount column (Quantity * UnitPrice)
df_clean['TotalAmount'] = df_clean['Quantity'] * df_clean['UnitPrice']

print("Cleaned Dataset Shape:", df_clean.shape)
df_clean.info()
df_clean.head()

Cleaned Dataset Shape: (397884, 9)
<class 'pandas.DataFrame'>
Index: 397884 entries, 0 to 541908
Data columns (total 9 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    397884 non-null  str           
 1   StockCode    397884 non-null  str           
 2   Description  397884 non-null  str           
 3   Quantity     397884 non-null  int64         
 4   InvoiceDate  397884 non-null  datetime64[us]
 5   UnitPrice    397884 non-null  float64       
 6   CustomerID   397884 non-null  int64         
 7   Country      397884 non-null  str           
 8   TotalAmount  397884 non-null  float64       
dtypes: datetime64[us](1), float64(2), int64(2), str(4)
memory usage: 30.4 MB


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TotalAmount
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34


In [5]:
# Save cleaned transaction dataset to CSV
df_clean.to_csv('../data/cleaned_online_retail.csv', index=False)
print("Saved cleaned dataset to data/cleaned_online_retail.csv successfully!")

Saved cleaned dataset to data/cleaned_online_retail.csv successfully!


In [6]:
# Define snapshot date as 1 day after the latest transaction date in dataset
snapshot_date = df_clean['InvoiceDate'].max() + dt.timedelta(days=1)

# Group by CustomerID and calculate RFM metrics
rfm = df_clean.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days, # Recency
    'InvoiceNo': 'nunique',                                # Frequency
    'TotalAmount': 'sum'                                   # Monetary
}).reset_index()

# Rename columns
rfm.columns = ['CustomerID', 'Recency', 'Frequency', 'Monetary']

print("RFM Table Shape:", rfm.shape)
rfm.head()

RFM Table Shape: (4338, 4)


,CustomerID,Recency,Frequency,Monetary
0,12346,326,1,77183.60
1,12347,2,7,4310.00
2,12348,75,4,1797.24
3,12349,19,1,1757.55
4,12350,310,1,334.40


In [7]:
# 1. Assign quartile scores (1 to 4) for Recency, Frequency, and Monetary
rfm['R_Score'] = pd.qcut(rfm['Recency'], 4, labels=[4, 3, 2, 1])  # Lower Recency = Better Score (4)
rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'), 4, labels=[1, 2, 3, 4])
rfm['M_Score'] = pd.qcut(rfm['Monetary'], 4, labels=[1, 2, 3, 4])

# 2. Combine scores to create RFM_Segment string and total RFM_Score
rfm['RFM_Segment'] = rfm['R_Score'].astype(str) + rfm['F_Score'].astype(str) + rfm['M_Score'].astype(str)
rfm['RFM_Score'] = rfm[['R_Score', 'F_Score', 'M_Score']].astype(int).sum(axis=1)

# 3. Define Churn target variable (1 = Churned if Recency > 90 days, 0 = Active)
rfm['Churn'] = np.where(rfm['Recency'] > 90, 1, 0)

print("Churn Distribution:")
print(rfm['Churn'].value_counts(normalize=True))
rfm.head()

Churn Distribution:
Churn
0    0.665975
1    0.334025
Name: proportion, dtype: float64


,CustomerID,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Segment,RFM_Score,Churn
0,12346,326,1,77183.60,1,1,4,114,6,1
1,12347,2,7,4310.00,4,4,4,444,12,0
2,12348,75,4,1797.24,2,3,4,234,9,0
3,12349,19,1,1757.55,3,1,4,314,8,0
4,12350,310,1,334.40,1,1,2,112,4,1


In [8]:
# Save RFM and Churn dataset to CSV
rfm.to_csv('../data/rfm_churn_data.csv', index=False)
print("Saved RFM dataset to data/rfm_churn_data.csv successfully!")

Saved RFM dataset to data/rfm_churn_data.csv successfully!


In [10]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, precision_score

# 1. Select Feature matrix (X) WITHOUT Recency/RFM_Score to prevent data leakage
# We use purchase frequency, monetary value, and avg transaction value
rfm['Avg_Order_Value'] = rfm['Monetary'] / rfm['Frequency']

X = rfm[['Frequency', 'Monetary', 'Avg_Order_Value']]
y = rfm['Churn']

# 2. Train-Test Split (80% training, 20% testing)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 3. Instantiate and train Random Forest Classifier
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# 4. Predict on the test set
y_pred = model.predict(X_test)

# 5. Print realistic evaluation metrics
print("--- Realistic Model Performance ---")
print(classification_report(y_test, y_pred))

# 6. Feature Importance
feature_importances = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False)
print("\n--- Feature Importances ---")
print(feature_importances)

--- Realistic Model Performance ---
              precision    recall  f1-score   support

           0       0.73      0.76      0.74       578
           1       0.47      0.43      0.45       290

    accuracy                           0.65       868
   macro avg       0.60      0.60      0.60       868
weighted avg       0.64      0.65      0.64       868


--- Feature Importances ---
Monetary           0.473088
Avg_Order_Value    0.420509
Frequency          0.106403
dtype: float64
